# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata information
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Cite as: {metadata.citeAs}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will retrieve all record sets (`@id`) and inspect their corresponding fields and their `@id`s.

In [ ]:
# List all available record sets and their fields by @id
record_sets = [rs for rs in dataset.record_sets()]
print(f"Number of record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record set name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s discovered above.

Below we demonstrate loading each record set, indexed by its `@id`, into a pandas DataFrame.

In [ ]:
# Extract data for every record set using its @id

record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"RecordSet @id: {rsid}, Shape: {dataframes[rsid].shape}")
# Display columns for the first record set as example
if record_set_ids:
    print(f"\nColumns for first record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We will choose a single record set and experiment with a numeric field, performing filtering, normalization, and grouping by another field. We will use the fields' `@id` values in all operations for full reproducibility.

Please adapt the field `@id` variables as needed for your specific analysis.

In [ ]:
# Example: EDA using first available record set with a numeric field
import numpy as np
# Identify a numeric field in the first record set
rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[rs_id] if rs_id is not None else None

# Attempt to infer a numeric field based on DataFrame dtypes
numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)] if df is not None else []
group_fields = [col for col in df.columns if df[col].dtype == 'object'] if df is not None else []

if not numeric_fields:
    print("No numeric fields found in this record set for EDA.")
else:
    numeric_field_id = numeric_fields[0]  # Choose the first numeric field by @id
    print(f"Using numeric field @id for EDA: {numeric_field_id}")
    threshold = df[numeric_field_id].dropna().mean() # Use mean as threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col_name = f"{numeric_field_id}_normalized"
    filtered_df[norm_col_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col_name]].head())

    # Choose a group field for grouping
    if group_fields:
        group_field_id = group_fields[0]
        print(f"\nGrouping by field @id: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_"+numeric_field_id)
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we create a histogram of a numeric field or a barplot of grouped field means, using `matplotlib`. You can adjust the field `@id`s as returned previously.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    numeric_field_id = numeric_fields[0]
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_fields:
        group_field_id = group_fields[0]
        plt.figure(figsize=(10,4))
        mean_vals = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        mean_vals.plot.bar()
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the FAIR² dataset and explored available record sets and their fields using their `@id`s.
- Data was extracted into pandas DataFrames for further manipulation and analysis.
- A numeric field was selected for exploratory data analysis, including filtering, normalization, and grouping; distributions and grouped means were visualized.
- For further steps, users are encouraged to consult the Croissant schema and documentation to target specific analytical questions, using the `@id` conventions established here for full reproducibility.